In [ ]:
import sys
from pathlib import Path
import numpy as np, cv2
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import preprocess as pp
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
rgb = pp.load_rgb(ROOT / "data/stratford/1958.png")
grey = pp.to_grey(rgb)
print(rgb.shape)
plt.figure(figsize=(8, 7)); plt.imshow(grey[::4, ::4], cmap="gray"); plt.show() ##this takes every 4th pixel to make it faster to show here and nothing is lost ca
##cause R=G=B here and converted from 3 to 1

In [ ]:
##check if its really grascale - validation check
print(np.array_equal(rgb[..., 0], rgb[..., 1]), np.array_equal(rgb[..., 1], rgb[..., 2]))

In [ ]:
invalid = pp.nodata_mask(rgb)                           # True where there's no data (big black areas + 3 px)
print(f"no data: {100 * invalid.mean():.1f}%  (info.txt: 6%)")  # mean of True/False = fraction True; x100 = %
show = np.dstack([grey] * 3)[::4, ::4].copy()           # stack grey 3 times -> an RGB image we can colour; every 4th pixel
show[invalid[::4, ::4]] = [255, 0, 0]                   # wherever the mask is True, paint the pixel red
plt.figure(figsize=(8, 7))                              # new figure
plt.imshow(show)                                        # draw it
plt.show()                                              # display

so wherever it came as 0, that was invalid and it was 6 percent out of the whole picture , but they only count as invalid when they belong to a black area of 500 px or more. That area is then grown by 3 px. A small black spot, like a shadow, stays valid. So the 6% is “big black areas plus their edges” out of the whole picture.

In [ ]:
# runs the script as if from Terminal: all 10 images -> data/processed/ + results/
%run ../src/preprocess.py

# the script switched plotting to file-only mode; switch inline display back on
%matplotlib inline

For Charlottetown, at least 54.8% of the block is usable for any pair of years you compare. That’s the worst case, because those pixels have data in every year.

Most comparisons get more than that. 1935 vs 2020, for example, only needs those two years to have data, so it’s well above 54.8%.

For Stratford, the same floor is 94%.

One quick check: the 1935, 1958 and 1968 numbers are identical in Stratford (958.8 ha), and 1935 and 1968 are identical in Charlottetown (3687.9 ha). That probably means the archive cut out the water with the same outline in each year.

In [ ]:
# load the saved masks (255 = valid, 0 = no data) for three Stratford years
m35 = cv2.imread(str(ROOT / "data/processed/stratford/1935_valid.png"), cv2.IMREAD_GRAYSCALE)
m58 = cv2.imread(str(ROOT / "data/processed/stratford/1958_valid.png"), cv2.IMREAD_GRAYSCALE)
m68 = cv2.imread(str(ROOT / "data/processed/stratford/1968_valid.png"), cv2.IMREAD_GRAYSCALE)
# True if every single pixel is the same -> the same cut-out shape was used
print(np.array_equal(m35, m58), np.array_equal(m58, m68))

In [ ]:
# make sure all three loaded (a wrong path gives None, which also prints False)
print(m35.shape, m58.shape, m68.shape)
# count pixels where the two masks disagree (one says valid, the other no data)
diff_35_58 = (m35 != m58).sum()
diff_58_68 = (m58 != m68).sum()
# show the count and the share of the whole image (3000 x 3400 px)
print(diff_35_58, f"{100 * diff_35_58 / m35.size:.3f}%")
print(diff_58_68, f"{100 * diff_58_68 / m58.size:.3f}%")

In [ ]:
# True if 1935 and 1968 masks are pixel-for-pixel identical
print(np.array_equal(m35, m68))
# row and column of every pixel where 1958 differs from 1935
rows, cols = np.nonzero(m35 != m58)
# the box around those pixels: where on the image the difference sits
print(rows.min(), rows.max(), cols.min(), cols.max())

In [ ]:
# clamp the start at 0 so a negative value can't wrap around to the end
ra, ca = max(r0 - 50, 0), max(c0 - 50, 0)
grey58 = pp.to_grey(pp.load_rgb(ROOT / "data/stratford/1958.png"))   # load 1958 as greyscale
fig, ax = plt.subplots(1, 2, figsize=(12, 8))                         # two plots side by side
ax[0].imshow(grey58[ra:r1 + 50, ca:c1 + 50], cmap="gray")             # left: the image in that corner
ax[0].set_title("1958 image")
ax[1].imshow((m35 != m58)[ra:r1 + 50, ca:c1 + 50], cmap="gray")       # right: white = mask differs
ax[1].set_title("mask differences")
plt.show()

That rules out my guess. 1935 and 1968 are exactly identical (True), but the 1958 differences aren’t one small patch. They’re scattered across a large box in the top-left corner, rows 5–1393 and columns 0–695. 652 pixels spread over that area means a thin sliver along the edge of the black no-data region, not a blob or an ink mark.

In [ ]:
fig.savefig(ROOT / "results/mask_edge_1958.png", dpi=150, bbox_inches="tight")   # save the figure as a PNG in results/; tight = trim extra white space
plt.show()